Model Cookbook
==============

The model cookbook provides a concise reference to lens model composition tools, specifically the `Model` and
`Collection` objects.

Examples using different PyAutoLens API’s for model composition are provided, which produce more concise and
readable code for different use-cases.

__Contents__

- **Simple Lens Model:** Compose a simple lens model with a lens galaxy and source galaxy.
- **More Complex Lens Models:** Extend the simple model to have multiple light or mass profiles and multiple galaxies.
- **Concise API:** Compose a lens model using the concise API, which is more readable and concise.
- **Prior Customization:** Customize the priors of individual lens model parameters using uniform, log-uniform and Gaussian priors.
- **Model Customization:** Customize the lens model parameters, including parameter pairing, fixing and offsets.
- **Redshift Free:** Make the redshift of a galaxy a free parameter in the model-fit.
- **Solved Parameters:** Parameters which are solved for during the fit, and parameters missing from your configuration.
- **Available Model Components:** List the available light profiles, mass profiles and other components that can be used for lens modeling.

Advanced Features:

**JSon Outputs:** Output a model to a .json file on hard-disk, which can be loaded and modified.
**Many Profile Models:** Compose and fit models with many light profiles, such as the Multi Gaussian Expansion (MGE) and shapelets.
**Model Linking:** Link the inferred model of one phase to the model in a non-linear search chain.
**Across Datasets:** Compose models where the same model component is used across multiple datasets, with certain parameters free to vary.
**Relations:** Compose models where the free parameter(s) vary according to a user-specified function.
**PyAutoFit API:** Use the PyAutoFit API to compose lens models in more advanced ways.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/start_here.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Simple Lens Model__

A simple lens model has a lens galaxy with a Sersic light profile, Isothermal mass profile and source galaxy with 
a Sersic light profile:

In [ ]:
# Lens:

bulge = af.Model(al.lp_linear.Sersic)
mass = af.Model(al.mp.Isothermal)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=mass,
)

# Source:

source = af.Model(al.Galaxy, redshift=1.0, bulge=al.lp_linear.SersicCore)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

The redshifts in the above model are used to determine which galaxy is the lens and which is the source.

The model `total_free_parameters` tells us the total number of free parameters (which are fitted for via a 
non-linear search), which in this case is 17 (6 from the lens `Sersic`, 5 from the lens `Isothermal` and 6 from the 
source `SersicCore`). The `intensity` of each linear light profile is *not* one of them: it is solved for by the 
inversion during every likelihood evaluation.

In [ ]:
print(f"Model Total Free Parameters = {model.total_free_parameters}")

If we print the `info` attribute of the model we get information on all of the parameters and their priors.

In [ ]:
print(model.info)

The same model can also be drawn as a figure, which shows its structure at a glance.

The figure is the **map** and `model.info` is the **legend**. The map shows the shape of the model: which component 
owns which parameter, and what state every parameter is in (free, fixed, shared with another component, related to 
one by an expression, solved during the fit or missing from your configuration). The legend gives the numbers: the 
prior on every parameter and the value of every fixed one. The figure therefore shows one thing `model.info` cannot, 
the dashed `intensity · solved` pill on each linear light profile, whose `intensity` is not a model parameter at all 
but is solved for by the inversion at every likelihood evaluation; the footer totals the model up as 17 sampled 
scalars and 2 parameters solved during fitting.

In [ ]:
af.ModelPlotter(model).figure()

__More Complex Lens Models__

The API above can be easily extended to compose lens models where each galaxy has multiple light or mass profiles:

In [ ]:
# Lens:

bulge = af.Model(al.lp_linear.Sersic)
disk = af.Model(al.lp_linear.Exponential)

mass = af.Model(al.mp.Isothermal)
shear = af.Model(al.mp.ExternalShear)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    disk=disk,
    mass=mass,
    shear=shear,
)

# Source:

bulge = af.Model(al.lp_linear.SersicCore)
disk = af.Model(al.lp_linear.ExponentialCore)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge, disk=disk)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

print(model.info)

The figure now draws four profile cards inside the lens galaxy's card and two inside the source galaxy's, which is 
the quickest way to check that a multi-profile model is composed the way you intended. Every linear light profile 
carries its own dashed `intensity · solved` pill, so the footer counts four parameters solved during fitting 
alongside the 29 sampled scalars.

In [ ]:
af.ModelPlotter(model).figure()

The use of the words `bulge`, `disk`, `mass` and `shear` above are arbitrary. They can be replaced with any name you
like, e.g. `bulge_0`, `bulge_1`, `mass_0`, `mass_1`, and the model will still behave in the same way.

The API can also be extended to compose lens models where there are multiple galaxies:

In [ ]:

bulge = af.Model(al.lp_linear.Sersic)
mass = af.Model(al.mp.Isothermal)

lens_0 = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=mass,
)

bulge = af.Model(al.lp_linear.Sersic)
mass = af.Model(al.mp.Isothermal)

lens_1 = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=mass,
)

# Source 0:

bulge = af.Model(al.lp_linear.SersicCore)

source_0 = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Source 1 :

bulge = af.Model(al.lp_linear.SersicCore)

source_1 = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Lens Model:

model = af.Collection(
    galaxies=af.Collection(
        lens_0=lens_0, lens_1=lens_1, source_0=source_0, source_1=source_1
    ),
)

print(model.info)

The two lens galaxies are identical in structure, and so are the two source galaxies, so the figure draws each pair 
once inside a dashed plate badged `2 components` rather than drawing four cards. Their parameters are badged 
`independent`: two separate priors with the same configuration, which is not the same thing as one shared prior.

In [ ]:
af.ModelPlotter(model).figure()

The above lens model consists of only two planes (an image-plane and source-plane), but has four galaxies in total.
This is because the lens galaxies have the same redshift and the source galaxies have the same redshift.

If we gave one of the lens galaxies a different redshift, it would be included in a third plane, and the model would
perform multi-plane ray tracing when the model-fit is performed.

__Concise API__

If a light or mass profile is passed directly to the `af.Model` of a galaxy, it is automatically assigned to be a
`af.Model` component of the galaxy.

This means we can write the model above comprising multiple light and mass profiles more concisely as follows (also
removing the comments reading Lens / Source / Overall Lens Model to make the code more readable):

In [ ]:
lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=al.lp_linear.Sersic,
    disk=al.lp_linear.Sersic,
    mass=al.mp.Isothermal,
    shear=al.mp.ExternalShear,
)

source = af.Model(
    al.Galaxy,
    redshift=1.0,
    bulge=al.lp_linear.SersicCore,
    disk=al.lp_linear.ExponentialCore,
)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))
print(model.info)

The concise API is a shorthand for writing a model, not a different model. Note that this stage gives the lens galaxy 
a `Sersic` bulge *and* a `Sersic` disk, so the two collapse into a single dashed plate badged `2 components`: the 
figure groups repeated sibling components rather than drawing two identical cards, and the `independent` badges say 
that each of the two has its own priors.

In [ ]:
af.ModelPlotter(model).figure()

__Prior Customization__

We can customize the priors of the lens model component individual parameters, using the following three types
of priors:

`UniformPrior`: The values of a parameter are randomly drawn between a `lower_limit` and `upper_limit`. For example,
  the effective radius of ellipitical Sersic profiles typically assumes a uniform prior between 0.0" and 30.0".

`LogUniformPrior`: Like a `UniformPrior` this randomly draws values between a `limit_limit` and `upper_limit`, but the
  values are drawn from a distribution with base 10. This is used for the `intensity` of a light profile, as the
  luminosity of galaxies follows a log10 distribution.

`GaussianPrior`: The values of a parameter are randomly drawn from a Gaussian distribution with a `mean` and width
 `sigma`. For example, the $y$ and $x$ centre values in a light profile typically assume a mean of 0.0" and a
 sigma of 0.3", indicating that we expect the profile centre to be located near the centre of the image.
 

In [ ]:
# Lens:

bulge = af.Model(al.lp_linear.Sersic)
bulge.sersic_index = af.TruncatedGaussianPrior(
    mean=4.0, sigma=1.0, lower_limit=1.0, upper_limit=8.0
)

mass = af.Model(al.mp.Isothermal)
mass.centre.centre_0 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.1, lower_limit=-0.5, upper_limit=0.5
)
mass.centre.centre_1 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.1, lower_limit=-0.5, upper_limit=0.5
)
mass.einstein_radius = af.UniformPrior(lower_limit=0.0, upper_limit=8.0)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=mass,
)

# Source

bulge = af.Model(al.lp_linear.SersicCore)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)
source.bulge.effective_radius = af.TruncatedGaussianPrior(
    mean=0.1, sigma=0.05, lower_limit=0.0, upper_limit=1.0
)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

print(model.info)

Customizing a prior does not change a parameter's state -- a parameter with a customized prior is still sampled -- so 
the map is unchanged by the customization above, and this figure is the same map as the simple lens model at the top 
of this cookbook. Print `model.info`, or call `af.ModelPlotter(model).figure(detail="priors")`, to read the numbers 
that did change.

In [ ]:
af.ModelPlotter(model).figure()

__Model Customization__

We can customize the lens model parameters in a number of different ways, as shown below:

In [ ]:
# Lens:

bulge = af.Model(al.lp_linear.Sersic)
disk = af.Model(al.lp_linear.Exponential)

# Parameter Pairing: Pair the centre of the bulge and disk together, reducing
# the complexity of non-linear parameter space by N = 2

bulge.centre = disk.centre

# Parameter Fixing: Fix the sersic_index of the bulge to a value of 4, reducing
# the complexity of non-linear parameter space by N = 1

bulge.sersic_index = 4.0

mass = af.Model(al.mp.Isothermal)

# Parameter Offsets: Make the mass model centre parameters the same value as
# the bulge / disk but with an offset.

mass.centre.centre_0 = bulge.centre.centre_0 + 0.1
mass.centre.centre_1 = bulge.centre.centre_1 + 0.1

shear = af.Model(al.mp.ExternalShear)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    disk=disk,
    mass=mass,
    shear=shear,
)

# Source:

bulge = af.Model(al.lp_linear.SersicCore)
disk = af.Model(al.lp_linear.ExponentialCore)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge, disk=disk)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

# Assert that the effective radius of the bulge is larger than that of the disk.
# (Assertions can only be added at the end of model composition, after all components
# have been brought together in a `Collection`.
model.add_assertion(
    model.galaxies.lens.bulge.effective_radius
    > model.galaxies.lens.disk.effective_radius
)

# Assert that the Einstein Radius is below 3.0":
model.add_assertion(model.galaxies.lens.mass.einstein_radius < 3.0)

print(model.info)

This is the stage where the figure earns its keep: the paired `centre` is drawn once on the `bulge` badged 
`shared ×2`, with a blue link from the `disk` that reuses it (`↗ bulge.centre`), the fixed `sersic_index` is a grey 
pill, and each assertion is a compact dashed-orange label naming both of its operands rather than a line traced 
across the figure. The mass profile's two offset `centre` components are a relation, but a relation on one component 
of a *tuple* parameter is not yet annotated on the tuple's single pill, so the mass `centre` is drawn as an ordinary 
sampled pill and `model.info` is the place to read it.

In [ ]:
af.ModelPlotter(model).figure()

__Redshift Free__

The redshift of a galaxy can be treated as a free parameter in the model-fit by using the following API:

In [ ]:
redshift = af.Model(al.Redshift)
redshift.redshift = af.UniformPrior(lower_limit=0.0, upper_limit=2.0)

lens = af.Model(al.Galaxy, redshift=redshift, mass=al.mp.Isothermal)

source = af.Model(al.Galaxy, redshift=1.0, bulge=al.lp_linear.SersicCore)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

print(model.info)

A free redshift is an ordinary sampled parameter, so it is drawn as an ordinary `redshift` pill on the lens galaxy's 
card. A *fixed* redshift is not drawn as a pill at all: it is printed under the galaxy's header as `redshift = 1.0`, 
which the source galaxy above shows, and it is the one number the figure puts on the map rather than in the legend.

In [ ]:
af.ModelPlotter(model).figure()

The model-fit will automatically enable multi-plane ray tracing and alter the ordering of the planes depending on the
redshifts of the galaxies.

NOTE: For strong lenses with just two planes (an image-plane and source-plane) the redshifts of the galaxies do not
impact the model-fit. You should therefore never make the redshifts free if you are only modeling a two-plane lens
system. This is because lensing calculations can be defined in arc-second coordinates, which do not change as a
function of redshift.

Redshifts should be made free when modeling three or more planes, as the mulit-plane ray-tracing calculations have an
obvious dependence on the redshifts of the galaxies which could be inferred by the model-fit.

__Solved Parameters__

Some parameters of a lens model are neither sampled by the non-linear search nor fixed by us: they are solved for 
during the fit, at every likelihood evaluation. The model below contains all three of them, a linear light profile, a 
pixelized source and a point source solved analytically:

In [ ]:
# Lens: a linear light profile, whose `intensity` is solved for by the inversion.

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=af.Model(al.lp_linear.Sersic),
    mass=af.Model(al.mp.Isothermal),
)

# Source: a pixelization, whose source `reconstruction` is solved for by the inversion.

pixelization = af.Model(
    al.Pixelization,
    mesh=af.Model(al.mesh.Delaunay, pixels=500, zeroed_pixels=0),
    regularization=af.Model(al.reg.ConstantSplit),
)

source = af.Model(al.Galaxy, redshift=1.0, pixelization=pixelization)

# Point source: a `PointSolved`, whose `centre` is solved for analytically.

point_source = af.Model(al.Galaxy, redshift=1.0, point=af.Model(al.ps.PointSolved))

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source, point_source=point_source)
)

print(model.info)

Three parameters in the figure carry a dashed `solved` pill: the `intensity` of the linear `Sersic` bulge, which the 
inversion solves for by linear algebra; the `reconstruction` of the `Pixelization`, which is the solved surface 
brightness of every source pixel (the only sampled parameter of a pixelization is its regularization `coefficient`); 
and the `centre` of the `PointSolved` point source, which is solved for analytically. **None of the three has any 
counterpart in the `model.info` printed above**: they are additional information the figure supplies, which is why 
they are drawn dashed, named in the legend as *solved during fitting*, and counted separately in the footer.

A fourth state the figure can draw is `missing`, in red, and it is a different thing again: a parameter for which no 
prior or value is configured anywhere, which `model.info` prints as `Prior Missing: Enter Manually or Add to Config` 
and which the fit cannot start without. No pill in this cookbook is `missing`, because this workspace's 
`config/priors` covers every component used here -- the `Delaunay` mesh's `areas_factor`, for example, is configured 
as a constant 0.5 and is therefore drawn as an ordinary grey fixed pill. Write your own profile class, or use one 
whose entry is absent from `config/priors`, and its parameters appear as red `missing` pills: **unset 
configuration**, not solved and not absent from the model. Absence from the figure would read as absence from the 
model, so `missing` is a state of its own.

In [ ]:
af.ModelPlotter(model).figure()

The figure and `model.info` group the same model differently, and for models with many profiles the correspondence 
between them is genuinely many-to-many. The figure partitions **by component**: a Multi Gaussian Expansion model 
built from two bases of thirty Gaussians shows two `30 components` plates, split because each basis holds its own 
`ell_comps` pair. `model.info` groups **per parameter**: it prints a single `0 - 59` block for `centre` spanning both 
figure plates, two separate ellipticity blocks, and individual `sigma` blocks. What does hold exactly is the 
contract between them: every element drawn on the figure resolves to a path or grouped paths in `model.info`, and 
every omission and every added annotation (`solved`, `missing`) is explicit.

__Available Model Components__

The light profiles, mass profiles and other components that can be used for lens modeling are given at the following
API documentation pages:

 - https://pyautolens.readthedocs.io/en/latest/api/light.html
 - https://pyautolens.readthedocs.io/en/latest/api/mass.html
 - https://pyautolens.readthedocs.io/en/latest/api/pixelization.html

__JSon Outputs__

After a model is composed, it can easily be output to a .json file on hard-disk in a readable structure:

In [ ]:
import os
import json

model_path = Path("path", "to", "model", "json")

os.makedirs(model_path, exist_ok=True)

model_file = Path(model_path, "model.json")

with open(model_file, "w+") as f:
    json.dump(model.dict(), f, indent=4)

We can load the model from its `.json` file.

In [ ]:
model = af.Model.from_json(file=model_file)

print(model.info)

The reloaded model prints the same `model.info`, and `af.ModelPlotter(model).figure()` draws it the same way with 
one exception: a component containing no free parameters at all -- here the `Delaunay` mesh and the whole 
`point_source` galaxy -- is written to the `.json` file as an instance rather than a model, so the reloaded figure 
folds each into a single fixed pill instead of drawing its own card.

This means in **PyAutoLens** one can write a model in a script, save it to hard disk and load it elsewhere, as well
as manually customize it in the .json file directory.

This is used for composing complex models of group scale lenses.

__Many Profile Models (Advanced)__

Features such as the Multi Gaussian Expansion (MGE) and shapelets compose models consisting of 50 - 500+ light
profiles.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autolens_workspace/blob/main/notebooks/modeling/features/multi_gaussian_expansion.ipynb
https://github.com/PyAutoLabs/autolens_workspace/blob/main/notebooks/modeling/features/shapelets.ipynb

__Model Linking (Advanced)__

When performing non-linear search chaining, the inferred model of one phase can be linked to the model.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autolens_workspace/blob/main/notebooks/imaging/advanced/guides/modeling/chaining.ipynb

__Across Datasets (Advanced)__

When fitting multiple datasets, model can be composed where the same model component are used across the datasets
but certain parameters are free to vary across the datasets.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autolens_workspace/blob/main/notebooks/multi_dataset/start_here.ipynb

__Relations (Advanced)__

We can compose models where the free parameter(s) vary according to a user-specified function 
(e.g. y = mx +c -> effective_radius = (m * wavelength) + c across the datasets.

The following example notebooks show how to compose and fit these models:

https://github.com/PyAutoLabs/autolens_workspace/blob/main/notebooks/multi_dataset/features/wavelength_dependence/modeling.ipynb

__PyAutoFit API__

**PyAutoFit** is a general model composition library which offers even more ways to compose lens models not
detailed in this cookbook.

The **PyAutoFit** model composition cookbooks detail this API in more detail:

https://pyautofit.readthedocs.io/en/latest/cookbooks/model.html
https://pyautofit.readthedocs.io/en/latest/cookbooks/multi_level_model.html

__Wrap Up__

This cookbook shows how to compose simple lens models using the `af.Model()` and `af.Collection()` objects.